In [ ]:

from pathlib import Path
import json

baseDir = Path("data")
modelName = "sentence-transformers/all-MiniLM-L6-v2"
minTextLength = 15
topicThreshold = 0.28  # Lower threshold to capture more relevant texts
fallbackMin = 0.24     # Lower fallback, keywords + embedding combo should work
maxTopics = 3          # Keep max 3 to prevent sprawl
batchSize = 128

# Load topics from single source of truth
topics_file = Path("../app/static/topics.json")
topics_data = json.loads(topics_file.read_text(encoding="utf-8"))
topicSpecs = [(t["label"], t["description"], set(t["keywords"])) for t in topics_data["topics"]]



In [ ]:
topicLabels = [label for label, _, _ in topicSpecs]
topicLabelTexts = [f"{label} — {desc}" for label, desc, _ in topicSpecs]


In [3]:

import re

URL_PATTERN = re.compile(r"https?://\S+")
RT_PATTERN = re.compile(r"^rt\s+@\w+:\s*", re.IGNORECASE)
MENTION_PATTERN = re.compile(r"@\w+")
MULTISPACE_PATTERN = re.compile(r"\s+")
CONTROL_CHARS_PATTERN = re.compile(r"[\u0000-\u001f\u007f-\u009f]")

def canonicalizeKey(text: str) -> str:
    text = text or ""
    text = URL_PATTERN.sub(" ", text)
    text = RT_PATTERN.sub("", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = CONTROL_CHARS_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text.lower()

def cleanForInference(text: str) -> str:
    text = text or ""
    text = URL_PATTERN.sub(" ", text)
    text = RT_PATTERN.sub("", text)
    text = CONTROL_CHARS_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text


In [24]:

# Build keyword map from topicSpecs (single source of truth)
TOPIC_KEYWORDS = {label: keywords for label, _, keywords in topicSpecs}

def containsTopicKeywords(text: str, topic: str) -> bool:
    """Check if text contains keywords for a topic"""
    if topic not in TOPIC_KEYWORDS:
        return False
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in TOPIC_KEYWORDS[topic])



In [25]:

import torch
from sentence_transformers import SentenceTransformer

def chooseDevice() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

device = chooseDevice()
topicModel = SentenceTransformer(modelName, device=device)
topicEmbeddings = topicModel.encode(
    topicLabelTexts,
    convert_to_tensor=True,
    normalize_embeddings=True,
)


In [ ]:

import numpy as np
from sentence_transformers import util

def predictTopicsBatch(texts):
    cleaned = [cleanForInference(t) for t in texts]
    embeddings = topicModel.encode(
        cleaned,
        convert_to_tensor=True,
        normalize_embeddings=True,
        batch_size=batchSize,
        show_progress_bar=False,
    )
    scores = util.dot_score(embeddings, topicEmbeddings).cpu().numpy()
    out = []
    for text_idx, row in enumerate(scores):
        order = np.argsort(-row)
        best_score = float(row[order[0]]) if len(order) else -1
        
        # Select topics above threshold
        candidates = [(i, float(row[i])) for i in order if float(row[i]) >= topicThreshold]
        
        # Boost score if keywords are present in original text
        original_text = texts[text_idx]
        keyword_boosted = []
        for topic_idx, score in candidates:
            topic = topicLabels[topic_idx]
            if containsTopicKeywords(original_text, topic):
                score = min(1.0, score + 0.25)  # Boost by 0.25 for domain-specific keyword matches
            keyword_boosted.append((topic_idx, score))
        
        # Re-sort by boosted scores
        keyword_boosted.sort(key=lambda x: -x[1])
        
        # If no candidates meet threshold, use fallback
        if not keyword_boosted and best_score >= fallbackMin and len(order):
            keyword_boosted = [(int(order[0]), best_score)]
        
        # Deduplicate: remove redundant topics that are too similar
        selected_topics = []
        for topic_idx, score in keyword_boosted:
            topic = topicLabels[topic_idx]
            # Check if this topic is too similar to already selected ones
            is_redundant = False
            for existing_idx in [topicLabels.index(t) for t in selected_topics]:
                # If cosine similarity between topics is too high, skip
                topic_similarity = float(util.dot_score(
                    topicEmbeddings[topic_idx:topic_idx+1],
                    topicEmbeddings[existing_idx:existing_idx+1]
                ))
                if topic_similarity > 0.75:  # Topics too similar
                    is_redundant = True
                    break
            
            if not is_redundant:
                selected_topics.append(topic)
                if len(selected_topics) >= maxTopics:
                    break
        
        out.append(selected_topics)
    return out

In [27]:

import pandas as pd
sampleSize = 2000

def build_sample() -> pd.DataFrame:
    chunks = []
    seen = set()
    for path in sorted(baseDir.glob("*/message_nodes.csv")):
        df = pd.read_csv(path)
        if "text" not in df.columns:
            continue
        df = df[["text"]].copy()
        df["text"] = df["text"].fillna("").astype(str)
        df = df[df["text"].str.len() >= minTextLength]
        if df.empty:
            continue
        df["normText"] = df["text"].apply(canonicalizeKey)
        df = df[df["normText"].map(bool)]
        df = df[~df["normText"].isin(seen)]
        if df.empty:
            continue
        seen.update(df["normText"].tolist())
        df["dataset"] = path.parent.name
        df = df.head(sampleSize)
        chunks.append(df)
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=["text", "normText", "dataset"])

sample = build_sample()
if sample.empty:
    print("No sample data available to inspect")
else:
    texts = sample["text"].tolist()
    sample_topics = predictTopicsBatch(texts)
    counts = pd.Series([tuple(t) for t in sample_topics]).value_counts()
    print("Sample topics (top 5):")
    print(counts.head(5))


Sample topics (top 5):
()                          7653
(population & identity,)     164
(street mobilization,)        59
(media & censorship,)         51
(national symbols,)           36
Name: count, dtype: int64


In [29]:
# Diagnostic: Analyze topic distribution and overlap

sample = build_sample()
if not sample.empty:
    texts = sample["text"].tolist()
    sample_topics = predictTopicsBatch(texts)
    
    # 1. Topic frequency
    topic_counts = {}
    for topics in sample_topics:
        for t in topics:
            topic_counts[t] = topic_counts.get(t, 0) + 1
    
    # 2. Topic combinations
    combos = {}
    for topics in sample_topics:
        key = tuple(sorted(topics)) if topics else ("none",)
        combos[key] = combos.get(key, 0) + 1
    
    # 3. Texts with no topics
    no_topic = sum(1 for t in sample_topics if not t)
    avg_topics = sum(len(t) for t in sample_topics) / len(sample_topics) if sample_topics else 0
    
    print("=" * 60)
    print("TOPIC MODELING DIAGNOSTICS")
    print("=" * 60)
    print(f"\nTexts with no topics: {no_topic} ({100*no_topic/len(sample_topics):.1f}%)")
    print(f"Average topics per text: {avg_topics:.2f}")
    print(f"\nTopic Frequency (Top 15):")
    for topic, count in sorted(topic_counts.items(), key=lambda x: -x[1])[:15]:
        print(f"  {topic:30} {count:4} ({100*count/len(sample_topics):5.1f}%)")
    
    print(f"\nTop Topic Combinations:")
    for combo, count in sorted(combos.items(), key=lambda x: -x[1])[:10]:
        combo_str = " + ".join(combo) if combo != ("none",) else "none"
        print(f"  {combo_str:45} {count:3} ({100*count/len(sample_topics):5.1f}%)")
    
    # 4. Topic similarity heatmap
    print(f"\nTopic Embedding Similarities (checking for redundancy):")
    sim_matrix = util.dot_score(topicEmbeddings, topicEmbeddings).cpu().numpy()
    high_sims = []
    for i in range(len(topicLabels)):
        for j in range(i+1, len(topicLabels)):
            sim = float(sim_matrix[i][j])
            if sim > 0.65:
                high_sims.append((topicLabels[i], topicLabels[j], sim))
    
    if high_sims:
        print(f"  Found {len(high_sims)} topic pairs with similarity > 0.65:")
        for t1, t2, sim in sorted(high_sims, key=lambda x: -x[2]):
            print(f"    {t1:25} ↔ {t2:25} ({sim:.3f})")
    else:
        print(f"  ✓ No highly similar topic pairs found (good!)")

TOPIC MODELING DIAGNOSTICS

Texts with no topics: 7653 (95.7%)
Average topics per text: 0.04

Topic Frequency (Top 15):
  population & identity           167 (  2.1%)
  street mobilization              64 (  0.8%)
  media & censorship               53 (  0.7%)
  national symbols                 36 (  0.5%)
  power & institutions             14 (  0.2%)
  geographic territory             10 (  0.1%)
  armed conflict                    5 (  0.1%)
  ideology & texts                  2 (  0.0%)
  electoral politics                2 (  0.0%)

Top Topic Combinations:
  none                                          7653 ( 95.7%)
  population & identity                         164 (  2.0%)
  street mobilization                            59 (  0.7%)
  media & censorship                             51 (  0.6%)
  national symbols                               36 (  0.5%)
  power & institutions                           13 (  0.2%)
  geographic territory                            9 (  0.1%)
  ar

In [30]:

import pandas as pd
from tqdm.auto import tqdm

messagePaths = sorted(baseDir.glob("*/message_nodes.csv"))
for path in messagePaths:
    df = pd.read_csv(path)
    if "text" not in df.columns:
        continue
    df["text"] = df["text"].fillna("").astype(str)
    df["normText"] = df["text"].apply(canonicalizeKey)
    unique = df.drop_duplicates(subset="normText")
    valid = unique[
        (unique["text"].str.len() >= minTextLength) & unique["normText"].astype(bool)
    ].copy()
    topicMap = {}
    if not valid.empty:
        keys = valid["normText"].tolist()
        texts = valid["text"].tolist()
        preds = []
        batch_desc = f"batches in {path.parent.name}"
        for i in tqdm(
            range(0, len(texts), batchSize),
            desc=batch_desc,
            leave=False,
            unit="batch",
        ):
            batch = texts[i : i + batchSize]
            preds.extend(predictTopicsBatch(batch))
        topicMap = dict(zip(keys, preds))
    post_desc = f"posts in {path.parent.name}"
    norm_texts = df["normText"].tolist()
    topics = []
    with tqdm(norm_texts, total=len(norm_texts), desc=post_desc, leave=False, unit="post") as post_bar:
        for key in post_bar:
            topics.append(topicMap.get(key, []) if key else [])
    df["topics"] = topics
    df = df.drop(columns=["normText"])
    df.to_csv(path, index=False)


batches in afdjugendbw:   0%|          | 0/141 [00:00<?, ?batch/s]

posts in afdjugendbw:   0%|          | 0/31704 [00:00<?, ?post/s]

batches in generationidentitaire:   0%|          | 0/101 [00:00<?, ?batch/s]

posts in generationidentitaire:   0%|          | 0/20016 [00:00<?, ?post/s]

batches in jungenationalisten:   0%|          | 0/112 [00:00<?, ?batch/s]

posts in jungenationalisten:   0%|          | 0/30254 [00:00<?, ?post/s]

batches in tricoloredelsangueitalico:   0%|          | 0/167 [00:00<?, ?batch/s]

posts in tricoloredelsangueitalico:   0%|          | 0/35936 [00:00<?, ?post/s]

In [31]:

import json
from tqdm.auto import tqdm

graphPaths = sorted(baseDir.glob("*/graph.json"))
for path in graphPaths:
    graph = json.loads(path.read_text(encoding="utf-8"))
    messages = graph.get("messages", [])
    if not messages:
        continue
    texts = [(m.get("text") or "").strip() for m in messages]
    keys = [canonicalizeKey(t) for t in texts]
    unique = {}
    for key, text in zip(keys, texts):
        if not key or len(text) < minTextLength or key in unique:
            continue
        unique[key] = text
    topicMap = {}
    if unique:
        uniq_keys = list(unique.keys())
        uniq_texts = [unique[k] for k in uniq_keys]
        preds = []
        batch_desc = f"batches in {path.parent.name}"
        for i in tqdm(
            range(0, len(uniq_texts), batchSize),
            desc=batch_desc,
            leave=False,
            unit="batch",
        ):
            batch = uniq_texts[i : i + batchSize]
            preds.extend(predictTopicsBatch(batch))
        topicMap = dict(zip(uniq_keys, preds))
    post_desc = f"posts in {path.parent.name}"
    message_iter = zip(messages, keys, texts)
    with tqdm(message_iter, total=len(messages), desc=post_desc, leave=False, unit="post") as post_bar:
        for msg, key, text in post_bar:
            msg["topics"] = topicMap.get(key, []) if key and len(text) >= minTextLength else []
    path.write_text(json.dumps(graph, ensure_ascii=False, indent=2), encoding="utf-8")


batches in afdjugendbw:   0%|          | 0/141 [00:00<?, ?batch/s]

posts in afdjugendbw:   0%|          | 0/31704 [00:00<?, ?post/s]

batches in generationidentitaire:   0%|          | 0/101 [00:00<?, ?batch/s]

posts in generationidentitaire:   0%|          | 0/20016 [00:00<?, ?post/s]

batches in jungenationalisten:   0%|          | 0/112 [00:00<?, ?batch/s]

posts in jungenationalisten:   0%|          | 0/30254 [00:00<?, ?post/s]

batches in tricoloredelsangueitalico:   0%|          | 0/167 [00:00<?, ?batch/s]

posts in tricoloredelsangueitalico:   0%|          | 0/35936 [00:00<?, ?post/s]

In [32]:

import shutil

staticRoot = Path("../app/static/data")
for datasetDir in sorted(baseDir.iterdir()):
    if not datasetDir.is_dir():
        continue
    targetDir = staticRoot / datasetDir.name
    targetDir.mkdir(parents=True, exist_ok=True)
    for filename in ("message_nodes.csv", "graph.json"):
        src = datasetDir / filename
        if not src.exists():
            continue
        dest = targetDir / filename
        shutil.copy2(src, dest)
        print(f"Copied {src} → {dest}")


Copied data/afdjugendbw/message_nodes.csv → ../app/static/data/afdjugendbw/message_nodes.csv
Copied data/afdjugendbw/graph.json → ../app/static/data/afdjugendbw/graph.json
Copied data/generationidentitaire/message_nodes.csv → ../app/static/data/generationidentitaire/message_nodes.csv
Copied data/generationidentitaire/graph.json → ../app/static/data/generationidentitaire/graph.json
Copied data/jungenationalisten/message_nodes.csv → ../app/static/data/jungenationalisten/message_nodes.csv
Copied data/jungenationalisten/graph.json → ../app/static/data/jungenationalisten/graph.json
Copied data/tricoloredelsangueitalico/message_nodes.csv → ../app/static/data/tricoloredelsangueitalico/message_nodes.csv
Copied data/tricoloredelsangueitalico/graph.json → ../app/static/data/tricoloredelsangueitalico/graph.json
